In [2]:
import pandas
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
inputdata = {}
inputdata = pandas.read_csv('finaldataset.csv', header=[0], index_col=0).to_dict()
inputdata = {k: v for k, v in inputdata.items() if v}

textdictionary = inputdata.get('text')

output = {"Text": []}

stop_words = set(stopwords.words('english'))

for text in textdictionary.values():
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\{[^}]*\}', '', text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    word_tokens = word_tokenize(text)
    filtered_words = [word for word in word_tokens if word not in stop_words and word.isalpha()]
    text = ' '.join(filtered_words)
    output['Text'].append(text)

results = pandas.DataFrame(output)
print(results)

results.to_csv('clean_final_dataset.csv', index=True, index_label="Index")

for text in textdictionary.values():
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\{[^}]*\}', '', text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    word_tokens = word_tokenize(text)
    filtered_words = [word for word in word_tokens if word not in stop_words and word.isalpha()]
    text = ' '.join(filtered_words)
    output['Text'].append(text)

results = pandas.DataFrame(output)
results.to_csv('cleaninput_final_data.csv', index=True, index_label="Index")

                                                    Text
0      say first foremost place everything awesome ch...
1      great cauliflower wings horrible service wante...
2      came happy hour dirty martinis made perfection...
3      came bawk urban roots recently great spot met ...
4      chicken waffles caught attention didnt disappo...
...                                                  ...
11210  went waitresses recommendations fried chicken ...
11211  first visit last slow service wrong orderd pri...
11212  today decided try many times passing ribs chic...
11213  poor customer service chicken fish shrimp supe...
11214  dad came town looking good spot biscuits gravy...

[11215 rows x 1 columns]


In [13]:

import pandas as pd
import numpy as np
import re

df = pd.read_csv('cleaninput_final_data.csv')

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

documents = df['Text'].astype(str).fillna('').str.strip().tolist()
documents = [doc for doc in documents if len(doc.split()) >= 3]
vectorizer = CountVectorizer(max_df=0.75, min_df=5, stop_words='english', lowercase=True, max_features=3000)

X = vectorizer.fit_transform(documents)

lda = LatentDirichletAllocation(n_components=8, random_state=42, learning_method='online',batch_size=256, max_iter=60, n_jobs=-1,evaluate_every=5)

lda.fit(X)


def print_topics(model, vectorizer, n_top_words=12):
    feature_names = vectorizer.get_feature_names_out()
    for idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"\nTopic {idx + 1}: {', '.join(top_words)}")


print(print_topics(lda, vectorizer))

Number of documents after filtering: 22428
Sample document (first 150 chars): say first foremost place everything awesome chicken perfect salad yup perfect oysters oui je suis travis bartenders best funky murals decor always put...

Topic 1: food, came, service, minutes, table, time, wait, got, ordered, didnt, took, server

Topic 2: chicken, sweet, waffles, good, gravy, breakfast, pie, fried, gumbo, potato, biscuits, southern

Topic 3: food, great, service, place, good, amazing, friendly, definitely, staff, delicious, best, love

Topic 4: food, order, place, like, time, dont, im, know, good, got, make, say

Topic 5: pizza, dish, deep, chicago, good, crust, wings, salad, great, place, like, really

Topic 6: chicken, fried, rice, good, halo, lumpia, place, savory, filipino, like, food, grilled

Topic 7: fried, chicken, cheese, mac, greens, good, grits, catfish, delicious, food, sides, ordered

Topic 8: chicken, hot, fries, sandwich, good, spicy, sauce, got, spice, like, really, mac
None
